# Procedimento para calcular população de bacias hidrossanitárias
##### Este documento define as etapas para a obtenção do número de habitantes inseridos na Área de Prestação de Serviços de bacias de esgotamento

Importação de bibliotecas:

In [52]:
import pandas as pd
import geopandas as gpd
import os

## 1º Passo: Importação dos dados
##### Setores Censitários: https://www.ibge.gov.br/estatisticas/sociais/trabalho/22827-censo-demografico-2022.html?edicao=41852&t=resultados 
Fazer download da malha de setores centiários por UF
##### Domicílios: https://www.ibge.gov.br/estatisticas/sociais/populacao/38734-cadastro-nacional-de-enderecos-para-fins-estatisticos.html?edicao=38891&t=resultados
Selecionar arquivos por município
##### APS encaminhado pela CORSAN; Bacias delimitadas pelo Analista. Coluna com o nome das bacias também deve ser especificado.

*Buscar código do município em https://www.ibge.gov.br/explica/codigos-dos-municipios.php

In [53]:
#Tapes
setores = gpd.read_file(r'C:\Users\gabriel.coimbra\Desktop\Meus arquivos\Caçapava do Sul\Arquivos Baixados\RS_setores_CD2022.gpkg')
domicilios = pd.read_csv(r'C:\Users\gabriel.coimbra\Desktop\Meus arquivos\Tapes\Material Online\DomiciliosIBGE\4321105.csv',delimiter = ';')
aps = gpd.read_file(r'C:\Users\gabriel.coimbra\Desktop\Meus arquivos\Tapes\Projetos Antigos\Material Bianca\Tapes RS\shapes\APS.shp')
bacias = gpd.read_file(r'C:\Users\gabriel.coimbra\Desktop\Meus arquivos\Tapes\Shapes Criados\Projeto2024\BaciasHidro_V02.gpkg')
coluna_nome_bacias = 'Nome'
caminho_exportacao = r'C:\Users\gabriel.coimbra\Desktop\Meus arquivos\Tapes\popdom_bacia_2022-comIgrejas.xlsx'
crs = "EPSG:31982"

## Funções auxiliares

In [54]:
#Funções auxiliares

#contagem de domicílios em cada setor na APS

def somar_extensao_polig(lines_gdf, polys_gdf, poly_id_col="Nome"):
    """
    Retorna um DataFrame com o comprimento (m e km) das linhas dentro de cada polígono.
    Requer que ambos estejam em um CRS projetado em METROS.
    """
    # Garantir colunas necessárias
    polys = polys_gdf[[poly_id_col, "geometry"]].copy()
    lines = lines_gdf[["geometry"]].copy()

    # Corrigir geometrias inválidas (se necessário)
    if hasattr(polys.geometry, "make_valid"):
        polys["geometry"] = polys.geometry.make_valid()
    else:
        polys["geometry"] = polys.buffer(0)

    # Interseção (recorta as linhas por polígono)
    inter = gpd.overlay(lines, polys, how="intersection")

    # Comprimento em metros
    inter["Extensão de Rede (m)"] = inter.geometry.length

    # Soma por polígono
    out = inter.groupby(poly_id_col, as_index=False)["Extensão de Rede (m)"].sum()
    
    return out

def contar_pontos_poligono(polygons, points, polygon_id_col="poly_id", predicate="intersects"):
    if polygons.crs != points.crs:
        points = points.to_crs(polygons.crs)

    # Garante coluna de ID
    if polygon_id_col not in polygons.columns:
        polygons = polygons.reset_index(drop=False).rename(columns={"index": polygon_id_col})

    joined = gpd.sjoin(points, polygons[[polygon_id_col, "geometry"]], predicate=predicate)
    counts = joined.groupby(polygon_id_col).size().rename("n_pontos").reset_index()
    
    result = polygons.merge(counts, on=polygon_id_col, how="left")
    result["n_pontos"] = result["n_pontos"].fillna(0).astype(int)
    
    return result

# Não é necessário mexer nisso abaixo

## 2º Passo: Tratamento dos dados

Conforme Diretriz Corsan (2025), o IBGE considera, para a densidade domiciliar, somente os domicílios particulares ocupados (v0007), o qual não representa a realidade das economias residenciais no cadastro da Corsan/Aegea. Portanto, deve-se recalcular a densidade domiciliar dos setores censitários. Para recalcular a densidade domiciliar, deve-se dividir a população (v0001) pelo total de domicílios particulares (v0003), gerando uma nova coluna “Densidade”.

In [55]:
setores['Densidade'] = setores['v0001']/setores['v0003']

É necessário filtrar os domicílios particulares (COD_ESPECIE = 1) e igrejas (COD_ESPECIE = 8) e transformar csv de domicílios em um arquivo georreferenciado

In [56]:
domparticular = domicilios[(domicilios['COD_ESPECIE'] == 1) |(domicilios['COD_ESPECIE'] == 8) ]
domparticular = gpd.GeoDataFrame(domparticular, geometry=gpd.points_from_xy(domparticular.LONGITUDE, domparticular.LATITUDE), crs="EPSG:4326")

Deve-se colocar tudo no mesmo CRS definido

In [57]:
domparticular = domparticular.to_crs(crs)
bacias = bacias.to_crs(crs)
aps = aps.to_crs(crs)
setores = setores.to_crs(crs) #convertendo pra sistema de coordenadas padrão

##### Intersecção entre setores e APS

In [58]:
setores_aps = gpd.clip(setores, aps) #interseção entre setores e APS
domparticular_aps = gpd.clip(domparticular, aps) #interseção entre domicílios e APS

## 3º Passo: Cálculo da população na APS
##### As etapas realizadas são:
- Intersecção entre Setores e APS
- Intersecção entre Domicílios e APS
- Contagem de domicílios em cada setor na APS
- Calculo da população

##### Contagem de domicílios em cada setor na APS

In [59]:
populacao_aps = contar_pontos_poligono(setores_aps, domparticular_aps)

##### Cálculo da população com a densidade e n_pontos criado

In [60]:
populacao_aps['População 2022'] = populacao_aps['Densidade']*populacao_aps['n_pontos']

pop = populacao_aps['População 2022'].sum()
print(f"A população total na APS em 2022 é de {pop:.0f}")
econ = len(domparticular_aps)
print(f"A população total na APS em 2022 é de {econ:.0f}")

A população total na APS em 2022 é de 13308
A população total na APS em 2022 é de 7738


## 4º Passo: Cálculo da população por bacia (2022)

Após delimitar as bacias para pelo menos 90% dos domicílios do IBGE (Censo 2022), deverá ser identificado quantos domicílios estão inseridos em cada bacia. As etapas realizadas são:
- Criação de camada com domicílios classificados por setor e bacia
- Criação de camada com bacias divididas em setores
- Calculo da população com base na densidade de cada domicílio dentro de cada setor dividido pela bacia
- Agrupamento dos valores por bacia, gerando a quantidade de população e domicílios por bacia

##### Intersecções entre domicílios, setores e bacias

In [61]:
camada_unida = gpd.sjoin(
    domparticular_aps,
    bacias, 
    predicate="intersects",
    how="left"
)
camada_unida = camada_unida.drop(columns=['index_right'], errors='ignore')

dompart_setores = gpd.sjoin(
    camada_unida,
    populacao_aps,  
    predicate="intersects",
    how="left"
)

bacias_setores = gpd.sjoin(
    populacao_aps,
    bacias,  
    predicate="intersects",
    how="left"
)

dompart_setores_filtrado = dompart_setores[[coluna_nome_bacias, 'CD_SETOR']]
bacias_setores_filtrado = bacias_setores[[coluna_nome_bacias, 'CD_SETOR','Densidade']]

##### Contagem da quantidade de vezes que uma combinação Setores Censitários + Bacia aparece

In [62]:
# Passo 1: Contar ocorrências de Nome + CD_SETOR na planilha de referência
contagem = (
    dompart_setores_filtrado
    .groupby([coluna_nome_bacias, 'CD_SETOR'])
    .size()
    .reset_index(name='Domicílios')
)

# Passo 2: Fazer merge com o DataFrame base
bacias_setores_filtrado = bacias_setores_filtrado.merge(contagem, on=[coluna_nome_bacias, 'CD_SETOR'], how='left')

# Passo 3: Substituir NaN por 0 (caso não tenha ocorrência)
bacias_setores_filtrado['Domicílios'] = bacias_setores_filtrado['Domicílios'].fillna(0).astype(int)

##### Cálculo da população por combinação Setores Censitários + Bacia

In [63]:
bacias_setores_filtrado['População'] = bacias_setores_filtrado['Domicílios']*bacias_setores_filtrado['Densidade']

##### Soma da população calculada por bacia

In [64]:
bacias_populacao = bacias_setores_filtrado[[coluna_nome_bacias,'Domicílios','População']].groupby(coluna_nome_bacias).sum()
bacias_populacao

,Domicílios,População
Nome,,
Bacia 01,196,484.202163
Bacia 02,95,210.791246
Bacia 03,766,1566.408446
Bacia 04,515,1023.451742
Bacia 06,387,663.069643
Bacia 08,961,1902.279894
Bacia 09,185,314.600592
Bacia 10,1102,1960.531925
Bacia 11,56,108.897959


##### Exportar excel final

In [65]:
bacias_populacao.to_excel(caminho_exportacao)

# Resultados

In [66]:
pop_aps = populacao_aps['População 2022'].sum()
print(f"A população total na APS em 2022 é de {pop_aps:.0f}")
dom_aps = len(domparticular_aps)
print(f"Os domicílios totais na APS em 2022 é de {dom_aps:.0f}")

resultado_dompop = bacias_populacao.copy()

resultado_dompop['Dom % bacias'] = resultado_dompop['Domicílios']/(resultado_dompop['Domicílios'].sum())
resultado_dompop['Dom % APS'] = resultado_dompop['Domicílios']/dom_aps
resultado_dompop['Pop % bacias'] = resultado_dompop['População']/(resultado_dompop['População'].sum())
resultado_dompop['Pop % APS'] = resultado_dompop['População']/pop_aps

display(resultado_dompop)

A população total na APS em 2022 é de 13308
Os domicílios totais na APS em 2022 é de 7738


,Domicílios,População,Dom % bacias,Dom % APS,Pop % bacias,Pop % APS
Nome,,,,,,
Bacia 01,196,484.202163,0.027980,0.025330,0.039626,0.036385
Bacia 02,95,210.791246,0.013562,0.012277,0.017251,0.015840
Bacia 03,766,1566.408446,0.109350,0.098992,0.128190,0.117705
Bacia 04,515,1023.451742,0.073519,0.066555,0.083756,0.076905
Bacia 06,387,663.069643,0.055246,0.050013,0.054264,0.049825
Bacia 08,961,1902.279894,0.137188,0.124192,0.155677,0.142943
Bacia 09,185,314.600592,0.026410,0.023908,0.025746,0.023640
Bacia 10,1102,1960.531925,0.157316,0.142414,0.160444,0.147321
Bacia 11,56,108.897959,0.007994,0.007237,0.008912,0.008183


# Extensão e Área das Bacias

In [67]:
eixolog = gpd.read_file(r'C:\Users\gabriel.coimbra\Desktop\Meus arquivos\Tapes\Aerolevantamento\Fase 7 - Restituicao Planimetrica Digital (MUB)\Shapes\EixoLogradouro.shp')
eixolog =eixolog.to_crs(crs)
bacias_area = bacias.to_crs(crs)

bacias_area_len = somar_extensao_polig(eixolog, bacias_area, poly_id_col="Nome")
bacias_area_len["Área (km²)"] = bacias_area.geometry.area / 10**6

bacias_area_len = bacias_area_len.set_index("Nome")

resultado_final = resultado_dompop.merge(
    bacias_area_len,
    left_index=True,
    right_index=True,
    how="left"
)

resultado_final = resultado_final[['Domicílios','Dom % bacias','Dom % APS','População','Pop % bacias','Pop % APS','Extensão de Rede (m)','Área (km²)']].transpose()

display(resultado_final)

Nome,Bacia 01,Bacia 02,Bacia 03,Bacia 04,Bacia 06,Bacia 08,Bacia 09,Bacia 10,Bacia 11,Bacia 12,Bacia 5,Bacia 5A,Bacia 7,Bacia 7A
Domicílios,196.000000,95.000000,766.000000,515.000000,387.000000,961.000000,185.000000,1102.000000,56.000000,712.000000,451.000000,474.000000,936.000000,169.000000
Dom % bacias,0.027980,0.013562,0.109350,0.073519,0.055246,0.137188,0.026410,0.157316,0.007994,0.101642,0.064383,0.067666,0.133619,0.024126
Dom % APS,0.025330,0.012277,0.098992,0.066555,0.050013,0.124192,0.023908,0.142414,0.007237,0.092013,0.058284,0.061256,0.120961,0.021840
População,484.202163,210.791246,1566.408446,1023.451742,663.069643,1902.279894,314.600592,1960.531925,108.897959,262.168170,778.548775,907.631706,1685.285531,351.547619
Pop % bacias,0.039626,0.017251,0.128190,0.083756,0.054264,0.155677,0.025746,0.160444,0.008912,0.021455,0.063714,0.074278,0.137919,0.028770
Pop % APS,0.036385,0.015840,0.117705,0.076905,0.049825,0.142943,0.023640,0.147321,0.008183,0.019700,0.058503,0.068202,0.126638,0.026416
Extensão de Rede (m),1162.170412,1020.649968,6300.484461,5119.465884,4049.028006,10557.436555,2570.806687,13625.977230,1246.265885,12764.993086,4832.317795,4638.627023,10391.034525,1517.447108
Área (km²),0.085547,0.737581,0.518728,0.193989,0.280816,0.072317,0.803393,0.879178,0.544287,0.085148,0.221211,0.071688,0.262186,0.668929


In [68]:
resultado_final.to_excel(r'C:\Users\gabriel.coimbra\Desktop\Meus arquivos\Tapes\inputpredim.xlsx')